# Decision Planner Comparison

Colab notebook for training and comparing seven learnable decision planning strategies on a small CICIDS sample.

In [1]:
!pip install numpy scikit-learn transformers -q

In [2]:
!git clone https://github.com/Lawapaul/AI_Agentic_DL.git

import os
import sys

repo_path = '/content/AI_Agentic_DL'
sys.path.append(repo_path)

Cloning into 'AI_Agentic_DL'...
remote: Enumerating objects: 538, done.
remote: Counting objects: 100% (303/303), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 538 (delta 148), reused 236 (delta 86), pack-reused 235 (from 1)
Receiving objects: 100% (538/538), 322.47 KiB | 3.32 MiB/s, done.
Resolving deltas: 100% (250/250), done.


AI_Agentic_DL  drive  sample_data


In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import numpy as np

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed/'

X = np.load(path + 'X_test.npy')[:20]
y = np.load(path + 'y_test.npy')[:20]

X = X.reshape(X.shape[0], -1)

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)

preds = model.predict(X)
probs = model.predict_proba(X)

In [8]:
from decision_planner.rule_based import RuleBasedPlanner
from decision_planner.confidence_based import ConfidenceBasedPlanner
from decision_planner.risk_based import RiskBasedPlanner
from decision_planner.hybrid import HybridPlanner
from decision_planner.rl_planner import RLPlanner
from decision_planner.llm_planner import LLMPlanner
from decision_planner.policy_based import PolicyBasedPlanner

ModuleNotFoundError: No module named 'decision_planner'

In [ ]:
attacks = ['Attack' if int(pred) != 0 else 'Normal Traffic' for pred in preds[:10]]
confidences = [float(max(prob)) for prob in probs[:10]]
severities = [
    'High' if conf > 0.8 else 'Medium' if conf > 0.6 else 'Low'
    for conf in confidences
]
targets = [
    'No Action' if attack == 'Normal Traffic' else 'Block' if conf > 0.85 else 'Alert' if conf > 0.6 else 'Monitor'
    for attack, conf in zip(attacks, confidences)
]

list(zip(attacks, confidences, severities, targets))[:3]

In [ ]:
rule = RuleBasedPlanner().fit(attacks, confidences, targets)
conf = ConfidenceBasedPlanner().fit(confidences, targets)
risk = RiskBasedPlanner().fit(confidences, severities, targets)
hybrid = HybridPlanner().fit(attacks, confidences, severities, targets)
rl = RLPlanner().fit(attacks, confidences, targets, epochs=10)
llm = LLMPlanner().fit(attacks, confidences, severities, targets)
policy = PolicyBasedPlanner().fit(attacks, severities, targets)

In [ ]:
results = []

for i in range(10):
    attack = attacks[i]
    conf_score = confidences[i]
    severity = severities[i]

    results.append({
        'index': i,
        'attack': attack,
        'confidence': conf_score,
        'severity': severity,
        'target_action': targets[i],
        'rule': rule.decide(attack, conf_score, severity),
        'confidence_based': conf.decide(attack, conf_score, severity),
        'risk': risk.decide(attack, conf_score, severity),
        'hybrid': hybrid.decide(attack, conf_score, severity),
        'rl': rl.decide(attack, conf_score, severity),
        'llm': llm.decide(attack, conf_score, severity),
        'policy': policy.decide(attack, conf_score, severity)
    })

    print(results[-1])

In [ ]:
import json

save_path = os.path.join(repo_path, 'experiments', 'results')
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, 'decision_results_all.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('Saved successfully')